# Utility Token Embedding Eğitimi

**Amaç:** Qwen2.5-7B-Instruct + 5 DefensiveToken modelinin win-rate'ini artırmak için,
model ağırlıklarına dokunmadan sadece N adet **UtilityToken embedding vektörünü** öğrenmek.

**Yöntem:** Soft Prompt Tuning — frozen model üzerinde sadece yeni token embedding'leri güncellenir.

**Eğitim verisi:** `davinci_003_outputs.json` — temiz AlpacaFarm instruction-output çiftleri

**Loss:** Cross-entropy(model çıktısı, davinci-003 referans çıktısı) — sadece output token'larında

---
**Gereksinimler:**
- A100 GPU (40GB)
- `defensivetokens.json` (projeden yüklenecek)
- `davinci_003_outputs.json` (projeden yüklenecek)

In [ ]:
# @title 1. Kurulum
!pip install transformers==4.45.2 accelerate torch tqdm -q

In [ ]:
# @title 2. Import'lar ve Konfigürasyon
import json
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

# ── Konfigürasyon ──────────────────────────────────────────────
TARGET_MODEL      = "Qwen/Qwen2.5-7B-Instruct"
N_DEFENSIVE       = 5          # mevcut defensive token sayısı
N_UTILITY         = 5          # öğrenilecek utility token sayısı
MAX_LENGTH        = 512        # token bazında max uzunluk
TRAIN_SAMPLES     = 1000       # eğitimde kullanılacak örnek sayısı
BATCH_SIZE        = 4
GRAD_ACCUM        = 4          # efektif batch = 16
EPOCHS            = 3
LR                = 5e-3
SEED              = 42
OUTPUT_JSON       = "utilitytokens.json"

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'yok'}")

In [ ]:
# @title 3. Dosya Yükleme
# Projenizdeki şu iki dosyayı bu hücreyi çalıştırmadan önce Colab'a yükleyin:
#   - src/official_stacks/defensivetoken/defensivetokens.json
#   - src/official_stacks/meta_secalign/data/davinci_003_outputs.json

from google.colab import files

print("defensivetokens.json dosyasını seçin:")
uploaded = files.upload()
assert "defensivetokens.json" in uploaded, "defensivetokens.json yüklenmedi!"

print("\ndavinci_003_outputs.json dosyasını seçin:")
uploaded2 = files.upload()
assert "davinci_003_outputs.json" in uploaded2, "davinci_003_outputs.json yüklenmedi!"

with open("defensivetokens.json", "r", encoding="utf-8") as f:
    defensive_token_data = json.load(f)

with open("davinci_003_outputs.json", "r", encoding="utf-8") as f:
    alpaca_data = json.load(f)

print(f"\nDefensiveToken vektör sayısı : {len(defensive_token_data[TARGET_MODEL])}")
print(f"AlpacaFarm örnek sayısı      : {len(alpaca_data)}")

In [ ]:
# @title 4. Model ve Tokenizer — DefensiveToken + UtilityToken Ekle
print("Qwen2.5-7B-Instruct yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, trust_remote_code=True)

# ── DefensiveToken'ları tokenizer'a ekle ──────────────────────
defensive_token_names = [f"[DefensiveToken{i}]" for i in range(N_DEFENSIVE)]
tokenizer.add_special_tokens({"additional_special_tokens": defensive_token_names})

# ── UtilityToken'ları tokenizer'a ekle ───────────────────────
utility_token_names = [f"[UtilityToken{i}]" for i in range(N_UTILITY)]
tokenizer.add_special_tokens(
    {"additional_special_tokens": defensive_token_names + utility_token_names}
)

# ── Defensive + Utility token ID'lerini al ────────────────────
defensive_ids = [tokenizer.convert_tokens_to_ids(t) for t in defensive_token_names]
utility_ids   = [tokenizer.convert_tokens_to_ids(t) for t in utility_token_names]
print(f"DefensiveToken IDs : {defensive_ids}")
print(f"UtilityToken IDs   : {utility_ids}")

# ── Chat template: defensive + utility token prefix ───────────
# Orijinal template'deki defensive prefix'e utility prefix'i ekliyoruz
DEF_PREFIX  = "".join(defensive_token_names)
UTIL_PREFIX = "".join(utility_token_names)
FULL_PREFIX = DEF_PREFIX + UTIL_PREFIX

# Hem eğitim hem eval için kullanılacak template
TRAIN_TEMPLATE = (
    "{%- if add_defensive_tokens %}\n"
    "{{- '" + FULL_PREFIX + "' }}\n"
    "{%- endif %}\n"
    "{%- for message in messages %}\n"
    "{{- '<|im_start|>' + message['role'] + '\\n' + message['content'] | trim + '\\n\\n<|im_end|>\\n' }}\n"
    "{%- endfor %}\n"
    "{%- if add_generation_prompt %}\n"
    "{{- '<|im_start|>assistant\\n' }}\n"
    "{%- endif %}\n"
)
tokenizer.chat_template = TRAIN_TEMPLATE

# ── Modeli yükle ve embedding katmanını genişlet ──────────────
model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.resize_token_embeddings(len(tokenizer))

# ── DefensiveToken embedding'lerini defensivetokens.json'dan yükle ─
defensive_vectors = torch.tensor(
    defensive_token_data[TARGET_MODEL], dtype=torch.float32
)  # (5, H)
with torch.no_grad():
    for i, did in enumerate(defensive_ids):
        model.get_input_embeddings().weight[did] = defensive_vectors[i].to(
            model.get_input_embeddings().weight.dtype
        )

# ── Tüm model parametrelerini dondur ──────────────────────────
model.requires_grad_(False)
model.gradient_checkpointing_enable()
model = model.to(device)
model.eval()

total_params  = sum(p.numel() for p in model.parameters())
print(f"\nModel yüklendi. Toplam parametre: {total_params / 1e9:.2f}B")
print(f"Tüm parametreler donduruldu.")

In [ ]:
# @title 5. UtilityToken Embedding'lerini Başlat

# "Yardımcı" anlamlı kelimelerin embedding ortalamasından başlat
# Rastgele başlatmadan çok daha iyi bir başlangıç noktası
seed_words = ["helpful", "certainly", "sure", "assist", "gladly",
              "absolutely", "happy", "pleasure", "provide", "support"]

embed_layer = model.get_input_embeddings()
hidden_size = embed_layer.weight.shape[1]

with torch.no_grad():
    seed_token_ids = [
        tid for word in seed_words
        for tid in tokenizer.encode(word, add_special_tokens=False)
    ]
    seed_token_ids = list(set(seed_token_ids))  # tekrarları kaldır
    seed_embeds = embed_layer.weight[seed_token_ids].float().mean(dim=0)  # (H,)

# N_UTILITY adet bağımsız, eğitilebilir vektör
# Küçük gürültü ekleyerek her birini farklı başlat
utility_embeddings = nn.Parameter(
    seed_embeds.unsqueeze(0).repeat(N_UTILITY, 1)
    + 0.01 * torch.randn(N_UTILITY, hidden_size)
)  # (N_UTILITY, H), requires_grad=True

optimizer = torch.optim.AdamW([utility_embeddings], lr=LR, weight_decay=0.0)

print(f"Utility embedding boyutu : ({N_UTILITY}, {hidden_size})")
print(f"Eğitilebilir parametre   : {utility_embeddings.numel():,}")
print(f"Seed kelimeler           : {seed_words[:5]}... ({len(seed_token_ids)} token)")

In [ ]:
# @title 6. Eğitim Verisi Hazırlama

class AlpacaSFTDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=512):
        self.samples = []
        skipped = 0
        for row in rows:
            instruction = str(row.get("instruction", "")).strip()
            user_input  = str(row.get("input", "")).strip()
            ref_output  = str(row.get("output", "")).strip()

            if not instruction or not ref_output:
                skipped += 1
                continue

            # Prompt: defensive + utility token prefix dahil
            messages = [{"role": "system", "content": instruction}]
            if user_input:
                messages.append({"role": "user", "content": user_input})

            prompt_str = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                add_defensive_tokens=True,  # chat template defensive+utility prefix ekler
            )

            # Tam metin: prompt + referans çıktı + EOS
            full_str = prompt_str + ref_output + tokenizer.eos_token

            full_ids   = tokenizer(full_str,   add_special_tokens=False, return_tensors="pt").input_ids[0]
            prompt_ids = tokenizer(prompt_str, add_special_tokens=False, return_tensors="pt").input_ids[0]

            if len(full_ids) > max_length:
                full_ids = full_ids[:max_length]

            prompt_len = min(len(prompt_ids), len(full_ids))

            # Label maskeleme: prompt kısmı -100, sadece çıktı token'ları öğretilir
            labels = full_ids.clone()
            labels[:prompt_len] = -100

            # Çıktı kısmı yoksa (truncate nedeniyle) atla
            if (labels != -100).sum() == 0:
                skipped += 1
                continue

            self.samples.append({"input_ids": full_ids, "labels": labels})

        print(f"Yüklenen örnek: {len(self.samples)} | Atlanan: {skipped}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Farklı uzunluktaki örnekleri pad'le ve batch oluştur."""
    max_len = max(s["input_ids"].shape[0] for s in batch)
    B = len(batch)

    input_ids      = torch.zeros(B, max_len, dtype=torch.long)
    labels         = torch.full((B, max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros(B, max_len, dtype=torch.long)

    for i, s in enumerate(batch):
        L = s["input_ids"].shape[0]
        input_ids[i, :L]      = s["input_ids"]
        labels[i, :L]         = s["labels"]
        attention_mask[i, :L] = 1

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


# Veriyi karıştır ve örnekle
random.shuffle(alpaca_data)
train_rows = alpaca_data[:TRAIN_SAMPLES]

dataset    = AlpacaSFTDataset(train_rows, tokenizer, max_length=MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate_fn, drop_last=True)

total_steps = math.ceil(len(dataset) / BATCH_SIZE) * EPOCHS
print(f"\nToplam adım: {total_steps}")

In [ ]:
# @title 7. Embedding Injection Yardımcı Fonksiyonu

def embed_with_utility(embed_layer, input_ids, utility_embeddings, utility_ids):
    """
    Frozen embedding layer'dan genel embedding'leri al,
    utility token pozisyonlarında eğitilebilir vektörleri enjekte et.

    Gradient akışı:
    - Frozen token pozisyonları: grad yok
    - Utility token pozisyonları: grad utility_embeddings'e akar
    """
    with torch.no_grad():
        # Tüm token'lar için frozen embedding'leri al
        embeds = embed_layer(input_ids).to(utility_embeddings.dtype)  # (B, L, H)

    for i, uid in enumerate(utility_ids):
        # uid'nin göründüğü pozisyonlar için float mask
        mask = (input_ids == uid).float().unsqueeze(-1)  # (B, L, 1)

        # Pozisyonda frozen'ı sıfırla, trainable'ı ekle:
        #   embeds_new = embeds + mask * utility_emb - mask * frozen
        #              = frozen * (1-mask) + utility_emb * mask
        # Gradient sadece utility_embeddings[i]'ye akar
        embeds = embeds + mask * utility_embeddings[i] - mask * embeds.detach()

    return embeds  # (B, L, H) — utility pozisyonlarında grad var

In [ ]:
# @title 8. Eğitim Döngüsü

# Cosine LR scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=LR * 0.1
)

embed_layer = model.get_input_embeddings()
step = 0
best_loss = float("inf")
best_embeddings = utility_embeddings.data.clone()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    epoch_steps = 0
    optimizer.zero_grad()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch_idx, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(device)
        labels         = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Utility embedding'leri enjekte edilmiş embedding tensörü oluştur
        embeds = embed_with_utility(
            embed_layer, input_ids,
            utility_embeddings.to(device),
            utility_ids
        )  # (B, L, H)

        # Frozen model üzerinden forward pass
        outputs = model(
            inputs_embeds=embeds.to(model.dtype),
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_loss  += loss.item() * GRAD_ACCUM
        epoch_steps += 1

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            # Gradient clipping — embedding'lerin kararsız büyümesini önle
            torch.nn.utils.clip_grad_norm_([utility_embeddings], max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1

        pbar.set_postfix(loss=f"{epoch_loss / epoch_steps:.4f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")

    avg_loss = epoch_loss / epoch_steps
    print(f"Epoch {epoch+1} tamamlandı — Ortalama loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_embeddings = utility_embeddings.data.clone()
        print(f"  ✓ En iyi model güncellendi (loss={best_loss:.4f})")

print(f"\nEğitim tamamlandı. En iyi loss: {best_loss:.4f}")

In [ ]:
# @title 9. UtilityToken Vektörlerini Kaydet
# Çıktı formatı defensivetokens.json ile birebir aynı

utility_vectors = best_embeddings.cpu().float().tolist()  # [[...], [...], ...]

output = {TARGET_MODEL: utility_vectors}

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Kaydedildi: {OUTPUT_JSON}")
print(f"Vektör sayısı : {len(utility_vectors)}")
print(f"Vektör boyutu : {len(utility_vectors[0])}")
print(f"Dosya boyutu  : {Path(OUTPUT_JSON).stat().st_size / 1024:.1f} KB")

# Google Drive'a da kaydet (opsiyonel)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(OUTPUT_JSON, "/content/drive/MyDrive/utilitytokens.json")

# Colab'dan indir
from google.colab import files
files.download(OUTPUT_JSON)
print("İndirme başlatıldı.")

In [ ]:
# @title 10. Sağlık Kontrolü — Örnek Çıktılar Karşılaştır
# Eğitimden önce ve sonra aynı soruya modelin nasıl yanıt verdiğini göster

from transformers import GenerationConfig

TEST_CASES = [
    {
        "instruction": "Translate the following text to French.",
        "input": "The weather is beautiful today."
    },
    {
        "instruction": "Write a short poem about the ocean.",
        "input": ""
    },
    {
        "instruction": "Summarize the following paragraph in one sentence.",
        "input": "The Apollo 11 mission, launched on July 16, 1969, successfully landed astronauts Neil Armstrong and Buzz Aldrin on the Moon on July 20, making them the first humans to walk on the lunar surface."
    },
]


def generate_response(model, tokenizer, instruction, user_input,
                      utility_embeddings, utility_ids, use_utility=True,
                      max_new_tokens=150):
    messages = [{"role": "system", "content": instruction}]
    if user_input:
        messages.append({"role": "user", "content": user_input})

    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        add_defensive_tokens=True,
    )

    input_ids = tokenizer(
        prompt_str, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)

    with torch.no_grad():
        if use_utility:
            embeds = embed_with_utility(
                model.get_input_embeddings(),
                input_ids,
                best_embeddings.to(device),
                utility_ids,
            )
            output = model.generate(
                inputs_embeds=embeds.to(model.dtype),
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            # generate inputs_embeds ile çalışırken çıktıdan yalnızca yeni token'ları al
            # (inputs_embeds length'i bilmiyoruz, tüm output decode ediyoruz)
            text = tokenizer.decode(output[0], skip_special_tokens=True)
        else:
            output = model.generate(
                input_ids=input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_tokens = output[0][input_ids.shape[1]:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return text.strip()


print("=" * 70)
for i, tc in enumerate(TEST_CASES):
    print(f"\n--- Test {i+1}: {tc['instruction'][:50]} ---")

    resp_without = generate_response(
        model, tokenizer, tc["instruction"], tc["input"],
        best_embeddings, utility_ids, use_utility=False
    )
    resp_with = generate_response(
        model, tokenizer, tc["instruction"], tc["input"],
        best_embeddings, utility_ids, use_utility=True
    )

    print(f"[DefenseOnly]  : {resp_without[:200]}")
    print(f"[Def+Utility]  : {resp_with[:200]}")
print("=" * 70)

## Sonraki Adım: Projeye Entegrasyon

İndirilen `utilitytokens.json` dosyasını projenize koyun:
```
src/official_stacks/utilitytokens/utilitytokens.json
```

Entegrasyon için `src/official_stacks/utilitytokens/core.py` yazılacak —
`defensivetoken/core.py` ile aynı API'ye sahip olacak:
- `load_utility_tokens()` → JSON'dan vektörleri yükle
- `prepare_utility_model(model_name, output_root)` → model üzerine uygula

Eval pipeline'ı (`run_qwen_alpaca_eval`) **değişmeyecek** —
sadece `model_name_or_path` artık `...-5DefensiveTokens-5UtilityTokens` olacak.